# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyinoluwaDon/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task Type:** Scoring / Ranking

**Explanation:** Rather than simple binary classification (is a page outdated or not?), this lane frames content refresh as a ranking and scoring task. Content teams have finite editorial bandwidth. Assigning a decay priority score to each URL produces an ordered backlog so editors update pages with the highest expected traffic recovery first.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import pandas as pd

# Load dataset locally or directly from raw GitHub URL when running in Colab
url = "https://raw.githubusercontent.com/IyinoluwaDon/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

if os.path.exists("../../data/raw/content_refresh_anonymized.csv"):
    path = "../../data/raw/content_refresh_anonymized.csv"
elif os.path.exists("data/raw/content_refresh_anonymized.csv"):
    path = "data/raw/content_refresh_anonymized.csv"
else:
    path = url

df = pd.read_csv(path)
print(f"Total candidate URLs to score: {len(df)}")

Total candidate URLs to score: 30000


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target / Proxy Definition:** `needs_refresh` (Binary proxy or Continuous Decay Score)

**Label Source:** Observed search performance data.

**Definition:**
- *Binary Label:* `1` if organic clicks dropped by >20% year-over-year while impressions remained stable (indicating position drift or falling CTR rather than loss of search volume/demand), otherwise `0`.
- *Continuous Proxy:* `click_loss_ratio = (past_clicks - current_clicks) / past_clicks` to quantify traffic loss decision-support.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Inspect dataset structure for target formulation
print("Available dataset columns:")
print(df.columns.tolist())
df.describe()

Available dataset columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,...,sessions_prev_30d,content_age_days,age_tier_order,days_since_last_update,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,trend_pct
count,27532.000000,27532.000000,27532.000000,22301.000000,22301.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,...,30000.000000,30000.00000,30000.000000,30000.000000,30000.000000,30000.00000,30000.000000,29875.000000,30000.000000,26612.000000
mean,158.882391,0.146954,0.485342,3107.760325,20665.277835,5200.366300,16.097333,49.942467,37.066633,35.937700,...,10.283000,256.16780,4.786533,46.098300,0.510733,16.34238,2.534520,18.212921,0.768196,-4.785969
std,1518.270825,0.285241,2.101560,1452.382598,10115.344042,16838.019547,75.076958,152.101430,107.069131,103.748185,...,42.578003,132.70793,0.790392,42.078709,3.279162,15.21679,8.310096,29.472768,7.429454,473.861780
min,0.000000,0.000000,0.000000,8.000000,40.000000,1.000000,0.000000,0.000000,1.000000,1.000000,...,0.000000,90.00000,3.000000,1.000000,0.000000,0.00000,0.000000,0.000000,0.000000,-100.000000
25%,0.000000,0.000000,0.000000,2413.000000,15644.000000,81.000000,0.000000,2.000000,2.000000,2.000000,...,1.000000,132.00000,4.000000,20.000000,0.000000,6.20000,0.000000,0.000000,0.000000,-62.600000
50%,10.000000,0.000000,0.000000,2877.000000,19116.000000,731.000000,1.000000,8.000000,7.000000,7.000000,...,2.000000,236.00000,5.000000,20.000000,0.070000,10.80000,0.000000,5.000000,0.000000,-33.500000
75%,20.000000,0.130000,0.000000,3666.000000,24011.000000,3615.250000,7.000000,33.000000,27.000000,27.000000,...,7.000000,333.00000,5.000000,104.000000,0.290000,22.30000,1.350000,23.530000,0.000000,0.000000
max,74000.000000,1.000000,100.360000,9546.000000,111158.000000,517715.000000,4178.000000,5998.000000,4345.000000,4913.000000,...,4247.000000,564.00000,6.000000,373.000000,100.000000,245.00000,100.000000,300.000000,300.000000,44900.000000


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary Success Metric:** Precision@50

**Business Defense:** Editorial teams have constrained capacity and can typically only rewrite 20 to 50 pages per sprint. Precision@50 measures what percentage of the top 50 ranked pages in the priority queue actually suffered true traffic decay and produce positive recovery when refreshed. Maximizing Precision@50 guarantees that editorial bandwidth is not wasted manually auditing healthy or non-recoverable content.

**Secondary Metrics:** Mean Reciprocal Rank (MRR) and ROC-AUC to evaluate global ranking quality across the entire candidate backlog.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Define target decay signal based on observed 30-day click trends
df['click_loss_pct'] = ((df['clicks_prev_30d'] - df['clicks_last_30d']) / (df['clicks_prev_30d'] + 1)) * 100
df['target_decay'] = ((df['click_loss_pct'] > 20) & (df['impressions_prev_30d'] >= 50)).astype(int)

top_k = 50
decay_count = df['target_decay'].sum()
print(f"Total decay candidate pages in dataset: {decay_count} out of {len(df)} ({decay_count/len(df):.1%})")
print(f"Target evaluation metric: Precision@{top_k} >= 0.80 for top-priority backlog queue")

Total decay candidate pages in dataset: 5574 out of 30000 (18.6%)
Target evaluation metric: Precision@50 >= 0.80 for top-priority backlog queue


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of Analysis:** 1 row = 1 unique content piece / URL slice (`content_id`).

The dataset contains 30,000 unique content rows evaluated across 44 behavioral, engagement, performance, and temporal features (such as 30-day click velocity, position tier, word count, and freshness metrics).

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



# Display dataset shape and core feature slice for 1 row = 1 URL
key_cols = ['content_id', 'content_type', 'content_age_days', 'clicks_prev_30d', 'clicks_last_30d', 'avg_position', 'ctr', 'trend_pct']
print(f"Dataframe Dimensions: {df.shape[0]} rows (pages), {df.shape[1]} columns (features)")
print("\nUnit of Analysis Sample (1 row = 1 unique content piece):")
df[key_cols].head(5)

Dataframe Dimensions: 30000 rows (pages), 46 columns (features)

Unit of Analysis Sample (1 row = 1 unique content piece):


,content_id,content_type,content_age_days,clicks_prev_30d,clicks_last_30d,avg_position,ctr,trend_pct
0,content_304f48230142,keyword article,187,13,2,10.6,0.76,-41.4
1,content_a1fb4e703a9e,keyword article,445,1,2,20.3,0.05,-57.7
2,content_9aa793d4d895,keyword article,141,3,1,36.5,0.09,-60.9
3,content_331d6c4de07b,keyword article,463,17,22,6.2,0.49,-13.8
4,content_d99b7a2d90ca,keyword article,263,2,10,44.0,0.13,-34.7


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why ML Beats Fixed Rules:**
A simple heuristic rule like *"refresh any page where content_age_days > 180"* fails because age alone does not cause traffic decay. High-authority evergreen pages retain top search positions for years without updates, while recently edited pages (`days_since_last_update` < 30) can still suffer traffic drops due to intent shifts or competitor updates.

An ML scoring model evaluates non-linear interactions across 44 features—combining position drift (`avg_position`), 30-day click velocity (`clicks_prev_30d` vs `clicks_last_30d`), search demand (`search_volume`), and user engagement (`engagement_rate`, `scroll_rate`) to distinguish true content decay from normal seasonal variance.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Evaluate precision of a simple fixed rule vs actual target decay
rule_flagged = df[df['content_age_days'] > 180]
rule_precision = rule_flagged['target_decay'].mean()

print(f"Pages flagged by fixed rule (content_age_days > 180): {len(rule_flagged)}")
print(f"Fixed rule precision: {rule_precision:.1%}")
print("Conclusion: Fixed rules produce high false-positive rates; ML ranking is required to capture multi-feature interaction.")

Pages flagged by fixed rule (content_age_days > 180): 17728
Fixed rule precision: 17.9%
Conclusion: Fixed rules produce high false-positive rates; ML ranking is required to capture multi-feature interaction.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.